In [1]:
# QuizBloom
# Molly Mishra

import os
import json
import time
import random
import urllib.parse
import urllib.request

import pandas as pd

from IPython.display import display, HTML, JSON
from google.colab import output

OPENTDB_API = "https://opentdb.com/api.php"
OPENTDB_TOKEN = "https://opentdb.com/api_token.php"
CATEGORY_IDS = {
    "Science": 17,
    "General Knowledge": 9,
    "Python": 18,
}
QUESTIONS_PER_ROUND = 5

PY_KEYWORDS = [
    "python", "programming", "code", "coding", "function", "variable",
    "algorithm", "loop", "syntax", "compile", "script", "software",
    "java", "html", "css", "sql", "javascript", "c++", "array",
    "string", "boolean", "integer", "operator", "class", "object"
]

# Fallback / top-up bank for Python (no large verified Python MCQ API exists)
PY_BANK = [
    ("Which keyword defines a function in Python?", ["function", "define", "def", "fun"], 2, "Python uses the def keyword."),
    ("Which data type stores key-value pairs?", ["List", "Dictionary", "Tuple", "Set"], 1, "Dictionaries store key-value pairs."),
    ("What is 2 * 3?", ["6", "8", "9", "5"], 1, "* is exponentiation: 2 to the power of 3."),
    ("Which symbol starts a Python comment?", ["//", "#", "/*", "--"], 1, "A hash starts a single-line comment."),
    ("Which method adds an item to the end of a list?", ["add()", "insertEnd()", "append()", "push()"], 2, "append() adds to the end."),
    ("What is len('Python')?", ["5", "6", "7", "8"], 1, "Python has six characters."),
    ("Which data type is immutable?", ["List", "Dictionary", "Set", "Tuple"], 3, "Tuples cannot be modified."),
    ("What is 10 // 3?", ["3", "3.33", "1", "4"], 0, "Floor division gives 3."),
    ("Which statement is used for a condition?", ["loop", "if", "repeat", "define"], 1, "if checks a condition."),
    ("Which function converts a value to an integer?", ["str()", "float()", "int()", "integer()"], 2, "int() converts to integer."),
    ("Which loop iterates over a sequence?", ["for", "select", "switch", "case"], 0, "for iterates over items."),
    ("What does break do inside a loop?", ["Skips one iteration", "Ends the loop", "Restarts the loop", "Pauses the program"], 1, "break exits the nearest loop."),
    ("Which operator checks equality?", ["=", "==", "!=", ":="], 1, "== compares values."),
    ("Which collection stores unique elements?", ["List", "Tuple", "Set", "String"], 2, "A set holds unique elements."),
    ("What is 'Py' + 'thon'?", ["Py thon", "Python", "Error", "Py+thon"], 1, "+ concatenates strings."),
    ("Which statement imports the math module?", ["include math", "using math", "import math", "load math"], 2, "Use import math."),
    ("What does bool(0) return?", ["True", "False", "0", "None"], 1, "Zero is falsy."),
    ("Which keyword sends a value back from a function?", ["send", "break", "return", "yieldValue"], 2, "return sends a value back."),
    ("What is 7 % 3?", ["1", "2", "3", "0"], 0, "The remainder is 1."),
    ("Which file mode opens a file for reading?", ["w", "a", "r", "x"], 2, "r opens for reading."),
]

_session = {"token": None, "py_pool": None, "py_used": set()}


def _http_json(url, timeout=20):
    req = urllib.request.Request(url, headers={"User-Agent": "QuizBloom/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))


def _get_token(reset=False):
    try:
        if reset and _session["token"]:
            _http_json(f"{OPENTDB_TOKEN}?command=reset&token={_session['token']}")
            return _session["token"]
        data = _http_json(f"{OPENTDB_TOKEN}?command=request")
        _session["token"] = data.get("token")
    except Exception:
        _session["token"] = None
    return _session["token"]


def _decode(s):
    return urllib.parse.unquote(s)


def _convert(item, source_label):
    correct = _decode(item["correct_answer"])
    options = [_decode(x) for x in item["incorrect_answers"]] + [correct]
    random.shuffle(options)
    return {
        "q": _decode(item["question"]),
        "o": options,
        "a": options.index(correct),
        "e": f"Difficulty: {item.get('difficulty', 'n/a')}. Source: {source_label}."
    }


_last_call = [0.0]


def _fetch(category_id, amount, tries=3):
    """Fetch from OpenTDB, respecting the 1 request / 5 s limit."""
    for _ in range(tries):
        wait = 5.3 - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)

        url = (f"{OPENTDB_API}?amount={amount}&category={category_id}"
               f"&type=multiple&encode=url3986")
        try:
            _last_call[0] = time.time()
            data = _http_json(url, timeout=10)
        except Exception:
            continue

        code = data.get("response_code")
        if code == 0:
            return data["results"]
        if code == 1:
            amount = max(1, amount // 2)
    return []


FALLBACK = {
    "Science": [
        ("Which planet is known as the Red Planet?", ["Venus", "Mars", "Jupiter", "Mercury"], 1, "Mars."),
        ("What is the chemical formula of water?", ["CO2", "O2", "H2O", "NaCl"], 2, "H2O."),
        ("What is the SI unit of force?", ["Joule", "Newton", "Watt", "Pascal"], 1, "Newton."),
        ("Which organ pumps blood around the body?", ["Liver", "Lungs", "Kidneys", "Heart"], 3, "The heart."),
        ("What is the central region of an atom called?", ["Electron", "Nucleus", "Shell", "Molecule"], 1, "The nucleus."),
        ("Which gas makes up most of Earth's atmosphere?", ["Oxygen", "Carbon dioxide", "Nitrogen", "Hydrogen"], 2, "Nitrogen (~78%)."),
        ("What is the basic unit of life?", ["Atom", "Organ", "Cell", "Tissue"], 2, "The cell."),
        ("Which planet is closest to the Sun?", ["Earth", "Venus", "Mars", "Mercury"], 3, "Mercury."),
    ],
    "General Knowledge": [
        ("What is the capital of Japan?", ["Beijing", "Seoul", "Tokyo", "Bangkok"], 2, "Tokyo."),
        ("Which is the largest ocean?", ["Atlantic", "Indian", "Arctic", "Pacific"], 3, "The Pacific."),
        ("What is the capital of Australia?", ["Sydney", "Melbourne", "Perth", "Canberra"], 3, "Canberra."),
        ("What is the largest mammal?", ["African elephant", "Giraffe", "Blue whale", "Hippopotamus"], 2, "The blue whale."),
        ("What is the square root of 144?", ["10", "11", "12", "14"], 2, "12 x 12 = 144."),
        ("Which Indian city is known as the Pink City?", ["Udaipur", "Jaipur", "Mumbai", "Chennai"], 1, "Jaipur."),
        ("Which is the largest hot desert?", ["Gobi", "Sahara", "Thar", "Kalahari"], 1, "The Sahara."),
        ("What is the smallest prime number?", ["0", "1", "2", "3"], 2, "Two."),
    ],
}


def _from_bank(rows, n):
    picked = random.sample(rows, min(n, len(rows)))
    out = []
    for q, o, a, e in picked:
        out.append({"q": q, "o": list(o), "a": a, "e": e + " (built-in question)"})
    return out


def get_questions(category):
    try:
        label = "Open Trivia DB (opentdb.com), CC BY-SA 4.0"

        if category == "Python":
            if _session["py_pool"] is None:
                raw = _fetch(CATEGORY_IDS["Python"], 50)
                pool = []
                for item in raw:
                    text = _decode(item["question"]).lower()
                    if any(k in text for k in PY_KEYWORDS):
                        pool.append(_convert(item, label))
                for q, o, a, e in PY_BANK:
                    pool.append({"q": q, "o": list(o), "a": a, "e": e})
                _session["py_pool"] = pool

            pool = _session["py_pool"]
            remaining = [i for i in range(len(pool)) if i not in _session["py_used"]]
            if len(remaining) < QUESTIONS_PER_ROUND:
                _session["py_used"] = set()
                remaining = list(range(len(pool)))
            chosen = random.sample(remaining, QUESTIONS_PER_ROUND)
            _session["py_used"].update(chosen)
            return {"success": True, "questions": [pool[i] for i in chosen]}

        raw = _fetch(CATEGORY_IDS[category], 10)
        seen = _session.setdefault("seen", set())
        fresh = [x for x in raw if x["question"] not in seen]
        if len(fresh) < QUESTIONS_PER_ROUND:
            fresh = raw          # allow repeats rather than fail
        fresh = fresh[:QUESTIONS_PER_ROUND]
        for x in fresh:
            seen.add(x["question"])

        if len(fresh) >= QUESTIONS_PER_ROUND:
            return {"success": True, "questions": [_convert(x, label) for x in fresh]}

        # API unavailable / rate-limited: use built-in questions
        return {"success": True, "questions": _from_bank(FALLBACK[category], QUESTIONS_PER_ROUND)}

    except Exception as e:
        try:
            return {"success": True, "questions": _from_bank(FALLBACK[category], QUESTIONS_PER_ROUND)}
        except Exception:
            return {"success": False, "error": str(e)}


# PYTHON BACKEND

RESULTS_FILE = "/content/quiz_results.csv"


def save_result(name, category, score, total):
    try:
        accuracy = round(score / total * 100, 2) if total else 0

        result = pd.DataFrame([{
            "Name": name,
            "Category": category,
            "Score": score,
            "Total Questions": total,
            "Accuracy (%)": accuracy
        }])

        if os.path.exists(RESULTS_FILE):
            result.to_csv(RESULTS_FILE, mode="a", header=False, index=False)
        else:
            result.to_csv(RESULTS_FILE, index=False)

        return {"success": True}

    except Exception as e:
        return {"success": False, "error": str(e)}


def get_leaderboard():
    try:
        if not os.path.exists(RESULTS_FILE):
            return {"success": True, "data": []}

        df = pd.read_csv(RESULTS_FILE)
        df = df.sort_values(
            by=["Accuracy (%)", "Score"],
            ascending=[False, False]
        ).head(10)

        return {"success": True, "data": df.to_dict(orient="records")}

    except Exception as e:
        return {"success": False, "error": str(e), "data": []}


def get_analytics():
    try:
        if not os.path.exists(RESULTS_FILE):
            return {"success": True, "games": 0, "average": 0, "best": 0}

        df = pd.read_csv(RESULTS_FILE)

        if df.empty:
            return {"success": True, "games": 0, "average": 0, "best": 0}

        return {
            "success": True,
            "games": len(df),
            "average": round(float(df["Accuracy (%)"].mean()), 2),
            "best": round(float(df["Accuracy (%)"].max()), 2)
        }

    except Exception as e:
        return {"success": False, "error": str(e)}


def _as_json(fn):
    """Return real JSON (application/json) so the browser can parse it."""
    def wrapper(*args):
        result = fn(*args)
        return JSON(json.loads(json.dumps(result, default=str)))
    return wrapper


output.register_callback("quiz.save", _as_json(save_result))
output.register_callback("quiz.leaderboard", _as_json(get_leaderboard))
output.register_callback("quiz.analytics", _as_json(get_analytics))
output.register_callback("quiz.questions", _as_json(get_questions))

# Frontend

html_code = r"""
<div id="color-quiz">

<style>
#color-quiz {
    font-family: Arial, Helvetica, sans-serif;
    color: #26324b;
    background: #ffffff;
    max-width: 1150px;
    margin: 15px auto;
    border-radius: 16px;
    overflow: hidden;
    box-shadow: 0 8px 35px rgba(40, 50, 90, 0.12);
}
#color-quiz * { box-sizing: border-box; }
#color-quiz button, #color-quiz input { font-family: inherit; }
#color-quiz button { cursor: pointer; transition: 0.2s ease; }
#color-quiz button:hover { transform: translateY(-2px); }
#color-quiz button:disabled { opacity: .6; cursor: not-allowed; transform: none; }

#color-quiz .navbar {
    display: flex; align-items: center; justify-content: space-between;
    gap: 15px; flex-wrap: wrap; padding: 18px 28px;
    background: white; border-bottom: 1px solid #edf0f7;
}
#color-quiz .logo { font-size: 24px; font-weight: 900; color: #0b3b46; letter-spacing: .3px; }
#color-quiz .nav-links { display: flex; flex-wrap: wrap; gap: 8px; }
#color-quiz .nav-btn {
    border: none; background: transparent; padding: 10px 12px;
    border-radius: 8px; color: #475569; font-weight: 700;
}
#color-quiz .nav-btn:hover { background: #e6f7f5; color: #0f766e; }

#color-quiz .hero {
    padding: 55px 30px;
    background: linear-gradient(120deg, #e3f6f3, #e4efff, #fff3df);
    display: grid; grid-template-columns: 1.5fr 1fr; gap: 25px; align-items: center;
}
#color-quiz .hero h1 {
    font-size: clamp(32px, 5vw, 50px); line-height: 1.15;
    margin: 12px 0; color: #0b3b46;
}
#color-quiz .hero p { line-height: 1.8; color: #596780; font-size: 16px; }
#color-quiz .badge {
    display: inline-block; padding: 8px 13px; background: #ffffff;
    color: #0d9488; border-radius: 30px; font-size: 12px; font-weight: 800;
}
#color-quiz .hero-art {
    text-align: center; padding: 28px 20px;
    background: rgba(255,255,255,.7); border-radius: 30px;
}
#color-quiz .hero-word { font-size: 38px; font-weight: 900; line-height: 1.25; letter-spacing: 1px; }
#color-quiz .hero-word.w1 { color: #0f766e; }
#color-quiz .hero-word.w2 { color: #1d4ed8; }
#color-quiz .hero-word.w3 { color: #b45309; }

#color-quiz .btn { border: none; border-radius: 10px; padding: 13px 20px; font-weight: 800; font-size: 14px; }
#color-quiz .btn-primary { color: white; background: #0d9488; box-shadow: 0 5px 12px rgba(13,148,136,.25); }
#color-quiz .btn-primary:hover { background: #0f766e; }
#color-quiz .btn-secondary { background: #e3f4f2; color: #0f766e; }
#color-quiz .btn-blue { background: #d6e6ff; color: #1d4ed8; }
#color-quiz .btn-green { background: #d8f9e7; color: #137c49; }
#color-quiz .btn-orange { background: #fff0dc; color: #a65b08; }

#color-quiz .section { padding: 35px 28px; }
#color-quiz .section-heading { text-align: center; margin-bottom: 25px; }
#color-quiz .section-heading h2 { font-size: 29px; margin: 0 0 10px; color: #0b3b46; }
#color-quiz .section-heading h2::after {
    content: ""; display: block; width: 54px; height: 4px; margin: 10px auto 0;
    border-radius: 4px; background: linear-gradient(90deg, #0d9488, #3b82f6);
}
#color-quiz .panel h2 { color: #0b3b46; }
#color-quiz .panel h3 { color: #0f766e; }
#color-quiz .result-box h2 { color: #0b3b46; }
#color-quiz .section-heading p { color: #778198; line-height: 1.6; margin: 0; }

#color-quiz .category-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 18px; }
#color-quiz .category-card { padding: 25px 20px; border-radius: 18px; border: 2px solid transparent; text-align: center; }
#color-quiz .python-card { background: #e6f0ff; border-color: #cfe0ff; }
#color-quiz .science-card { background: #e3f8ee; border-color: #cef0dd; }
#color-quiz .general-card { background: #fff0df; border-color: #ffe3c3; }
#color-quiz .category-icon {
    width: 64px; height: 64px; margin: 0 auto 14px; border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 22px; font-weight: 900; letter-spacing: .5px;
}
#color-quiz .python-card .category-icon { background: #cfe0ff; color: #1d4ed8; }
#color-quiz .science-card .category-icon { background: #c9f0dc; color: #15803d; }
#color-quiz .general-card .category-icon { background: #ffe0b8; color: #b45309; }
#color-quiz .python-card h3 { color: #1e40af; }
#color-quiz .science-card h3 { color: #166534; }
#color-quiz .general-card h3 { color: #9a4a07; }
#color-quiz .category-card h3 { font-size: 20px; margin: 5px 0 10px; }
#color-quiz .category-card p { font-size: 13px; line-height: 1.7; color: #68738a; min-height: 45px; }
#color-quiz .category-card .btn { width: 100%; margin-top: 10px; }

#color-quiz .features { display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; }
#color-quiz .feature { padding: 22px; border: 1px solid #e8ebf3; border-radius: 15px; background: white; }
#color-quiz .feature-icon { font-size: 24px; font-weight: 900; color: #0d9488; }
#color-quiz .feature h3 { margin: 12px 0 8px; color: #0f4c5c; }
#color-quiz .feature p { color: #778198; font-size: 13px; line-height: 1.7; }

#color-quiz .panel { padding: 26px; background: white; border: 1px solid #e6eaf2; border-radius: 18px; margin: 25px 0; }
#color-quiz .hidden { display: none !important; }
#color-quiz .muted { color: #7b8498; line-height: 1.7; }
#color-quiz .input-label { display: block; margin: 18px 0 8px; font-size: 14px; font-weight: 800; }
#color-quiz input {
    width: 100%; padding: 13px; border: 1px solid #cfd8e6; border-radius: 10px;
    font-size: 15px; outline: none;
    background: #ffffff !important;
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937;
    caret-color: #0d9488;
}
#color-quiz input::placeholder { color: #8a94a6; -webkit-text-fill-color: #8a94a6; opacity: 1; }
#color-quiz input:focus { border-color: #0d9488; box-shadow: 0 0 0 3px rgba(13,148,136,.15); }

#color-quiz .progress-track { height: 9px; background: #edf0f7; border-radius: 20px; overflow: hidden; margin: 18px 0 25px; }
#color-quiz .progress-bar { height: 100%; background: linear-gradient(90deg, #0d9488, #3b82f6); width: 0; transition: width .3s; }
#color-quiz .question { font-size: 23px; line-height: 1.5; font-weight: 800; margin: 20px 0; }

#color-quiz .option {
    display: flex; align-items: center; gap: 12px; width: 100%; text-align: left;
    border: 2px solid #e6eaf2; background: #fff; border-radius: 12px; padding: 14px;
    margin: 10px 0; color: #33415c; font-size: 15px; line-height: 1.5;
}
#color-quiz .option:hover { border-color: #5fc9bd; background: #f3fbfa; }
#color-quiz .option.selected { border-color: #0d9488; background: #e6f7f5; }
#color-quiz .option.correct { border-color: #2eb67d; background: #e0f8eb; }
#color-quiz .option.wrong { border-color: #ee7581; background: #fff0f0; }
#color-quiz .option-letter {
    display: inline-flex; align-items: center; justify-content: center;
    width: 31px; height: 31px; flex-shrink: 0; background: #e3f4f2;
    border-radius: 8px; font-weight: 900; color: #0f766e;
}

#color-quiz .feedback { margin-top: 16px; padding: 14px; border-radius: 10px; line-height: 1.7; }
#color-quiz .good { background: #e0f8eb; color: #137c49; }
#color-quiz .bad { background: #fff0f0; color: #a62d40; }
#color-quiz .row { display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 12px; }

#color-quiz .result-box { text-align: center; background: linear-gradient(130deg, #e3f6f3, #e4efff); border-radius: 18px; padding: 30px 20px; }
#color-quiz .result-number { font-size: 55px; font-weight: 900; color: #0d9488; margin: 10px 0; }
#color-quiz .stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin: 20px 0; }
#color-quiz .stat { padding: 18px 10px; text-align: center; background: #f7f8fc; border-radius: 12px; }
#color-quiz .stat strong { display: block; color: #0d9488; font-size: 24px; margin-bottom: 5px; }
#color-quiz .stat span { font-size: 12px; color: #778198; }

#color-quiz .table-wrap { overflow-x: auto; }
#color-quiz table { width: 100%; border-collapse: collapse; font-size: 13px; margin-top: 15px; }
#color-quiz th, #color-quiz td { text-align: left; padding: 12px 9px; border-bottom: 1px solid #e8ebf3; }
#color-quiz th { color: #0d9488; background: #effaf8; }
#color-quiz .error { color: #a62d40; background: #fff0f0; padding: 12px; margin-top: 12px; border-radius: 10px; }

#color-quiz .footer { background: #28324a; color: white; padding: 28px; text-align: center; line-height: 1.8; }
#color-quiz .footer small { color: #cbd2e3; }

@media (max-width: 720px) {
    #color-quiz .hero { grid-template-columns: 1fr; padding: 35px 22px; }
    #color-quiz .hero-word { font-size: 28px; }
    #color-quiz .category-grid, #color-quiz .features { grid-template-columns: 1fr; }
    #color-quiz .section { padding: 28px 18px; }
    #color-quiz .navbar { padding: 16px; }
    #color-quiz .nav-links { width: 100%; justify-content: space-between; }
    #color-quiz .nav-btn { padding: 8px; font-size: 12px; }
    #color-quiz .stats { gap: 7px; }
    #color-quiz .stat strong { font-size: 20px; }
    #color-quiz .panel { padding: 17px; }
    #color-quiz .question { font-size: 20px; }
}
</style>


<!-- NAVIGATION -->
<header class="navbar">
    <div class="logo">QuizBloom</div>
    <nav class="nav-links">
        <button class="nav-btn" data-page="home">Home</button>
        <button class="nav-btn" data-page="categories">Categories</button>
        <button class="nav-btn" data-page="leaderboard">Leaderboard</button>
    </nav>
</header>


<!-- HOME PAGE -->
<section id="cq-home">
    <div class="hero">
        <div>
            <span class="badge">LEARN • PLAY • GROW</span>
            <h1>Learning becomes fun when you play!</h1>
            <p>
                Test your knowledge, explore interesting topics,
                and challenge yourself with thousands of fresh questions.
                Every question is a new opportunity to learn.
            </p>
            <button class="btn btn-primary" data-page="categories">Explore Quizzes →</button>
        </div>
        <div class="hero-art"><div class="hero-word w1">Learn</div><div class="hero-word w2">Play</div><div class="hero-word w3">Grow</div></div>
    </div>

    <div class="section" id="cq-categories">
        <div class="section-heading">
            <h2>Explore Our Quiz Categories</h2>
            <p>Choose your favourite topic and start learning!</p>
        </div>

        <div class="category-grid">
            <div class="category-card python-card">
                <div class="category-icon">Pr</div>
                <h3>Programming</h3>
                <p>Explore Programming basics, data types, functions, operators, and loops.</p>
                <button class="btn btn-blue" data-category="Python">Start Programming Quiz</button>
            </div>

            <div class="category-card science-card">
                <div class="category-icon">Sc</div>
                <h3>Science</h3>
                <p>Discover facts about the human body, planets, physics, and nature.</p>
                <button class="btn btn-green" data-category="Science">Start Science Quiz</button>
            </div>

            <div class="category-card general-card">
                <div class="category-icon">GK</div>
                <h3>General Knowledge</h3>
                <p>Challenge yourself with geography, mathematics, and interesting facts.</p>
                <button class="btn btn-orange" data-category="General Knowledge">Start GK Quiz</button>
            </div>
        </div>
    </div>

    <div class="section" style="background:#f8f9ff;">
        <div class="section-heading">
            <h2>Why QuizBloom?</h2>
            <p>A simple and enjoyable way to test your knowledge.</p>
        </div>

        <div class="features">
            <div class="feature">
                <div class="feature-icon">01</div>
                <h3>Test Yourself</h3>
                <p>Answer multiple-choice questions and check your understanding.</p>
            </div>
            <div class="feature">
                <div class="feature-icon">02</div>
                <h3>Track Progress</h3>
                <p>View your scores and accuracy after every quiz.</p>
            </div>
            <div class="feature">
                <div class="feature-icon">03</div>
                <h3>Improve Every Day</h3>
                <p>Practise different categories and aim for a higher score.</p>
            </div>
        </div>
    </div>
</section>


<!-- QUIZ PAGE -->
<section id="cq-quiz" class="section hidden">
    <div class="panel">
        <div class="row">
            <div>
                <span class="badge" id="cq-category-label">Quiz Category</span>
                <h2 style="margin-bottom:5px;">Let's test your knowledge!</h2>
                <p class="muted" style="margin-top:0;">Read each question carefully before selecting an answer.</p>
            </div>
            <button class="btn btn-secondary" data-page="home">← Home</button>
        </div>

        <label class="input-label" for="cq-player-name">Your Name</label>
        <input id="cq-player-name" maxlength="40" placeholder="Enter your name">

        <div id="cq-question-area" class="hidden">
            <div class="row" style="margin-top:25px;">
                <strong id="cq-question-number">Question 1 of 5</strong>
                <span class="muted" id="cq-score-display">Score: 0</span>
            </div>

            <div class="progress-track"><div id="cq-progress" class="progress-bar"></div></div>

            <div id="cq-question" class="question"></div>
            <div id="cq-options"></div>
            <div id="cq-feedback" class="hidden"></div>

            <div class="row" style="margin-top:20px;">
                <button id="cq-check" class="btn btn-primary">Check Answer</button>
                <button id="cq-next" class="btn btn-primary hidden">Next Question →</button>
            </div>
        </div>

        <div id="cq-start-area">
            <p class="muted">Enter your name and press Begin. Questions are loaded live from the Open Trivia Database.</p>
            <button class="btn btn-primary" id="cq-begin">Begin Quiz →</button>
        </div>

        <div id="cq-error" class="error hidden"></div>
    </div>
</section>


<!-- RESULTS PAGE -->
<section id="cq-results" class="section hidden">
    <div class="section-heading">
        <h2>Your Quiz Results</h2>
        <p>Every attempt helps you learn something new.</p>
    </div>

    <div class="result-box">
        <h2 id="cq-result-title">Quiz Completed!</h2>
        <div class="result-number" id="cq-result-score">0/5</div>
        <p id="cq-result-message"></p>
        <p class="muted" id="cq-result-category"></p>
    </div>

    <div class="stats">
        <div class="stat"><strong id="cq-correct">0</strong><span>Correct Answers</span></div>
        <div class="stat"><strong id="cq-wrong">0</strong><span>Incorrect Answers</span></div>
        <div class="stat"><strong id="cq-accuracy">0%</strong><span>Accuracy</span></div>
    </div>

    <div class="panel">
        <h3>What would you like to do next?</h3>
        <div class="row" style="justify-content:flex-start;">
            <button class="btn btn-primary" id="cq-play-again">Play Again</button>
            <button class="btn btn-secondary" data-page="categories">Try Another Category</button>
            <button class="btn btn-secondary" data-page="leaderboard">View Leaderboard</button>
        </div>
        <p id="cq-save-status" class="muted"></p>
    </div>
</section>


<!-- LEADERBOARD PAGE -->
<section id="cq-leaderboard" class="section hidden">
    <div class="section-heading">
        <h2>Quiz Leaderboard</h2>
        <p>See the best results saved in this Colab session.</p>
    </div>

    <div class="panel">
        <div class="row">
            <h3>Top Players</h3>
            <button class="btn btn-secondary" id="cq-refresh">Refresh Results</button>
        </div>
        <div id="cq-leaderboard-table"><p class="muted">Loading results...</p></div>
    </div>

    <div class="panel">
        <h3>Quiz Statistics</h3>
        <div class="stats">
            <div class="stat"><strong id="cq-games">0</strong><span>Games Played</span></div>
            <div class="stat"><strong id="cq-average">0%</strong><span>Average Accuracy</span></div>
            <div class="stat"><strong id="cq-best">0%</strong><span>Best Accuracy</span></div>
        </div>
    </div>
</section>


<!-- FOOTER -->
<footer class="footer">
    <div style="font-size:23px;font-weight:900;">QuizBloom</div>
    <div>Learn something new with every question.</div>
</footer>


<script>
(function () {
    const root = document.getElementById("color-quiz");
    if (!root) return;

    const $ = id => document.getElementById("cq-" + id);

    const LOCAL_RESULTS = "quizbloom_results_v1";

    let category = "Python";
    let player = "";
    let questions = [];
    let index = 0;
    let score = 0;
    let selected = null;
    let checked = false;
    let finished = false;
    let loading = false;

    function readStorage(key, fallback) {
        try {
            const value = localStorage.getItem(key);
            return value ? JSON.parse(value) : fallback;
        } catch (e) {
            return fallback;
        }
    }

    function writeStorage(key, value) {
        try { localStorage.setItem(key, JSON.stringify(value)); } catch (e) {}
    }

    function page(name) {
        ["home", "quiz", "results", "leaderboard"].forEach(p => {
            $(p).classList.toggle("hidden", p !== name);
        });

        if (name === "leaderboard") refreshDashboard();
        if (name === "categories") {
            $("home").classList.remove("hidden");
            $("categories").scrollIntoView({behavior: "smooth", block: "start"});
        }
    }

    function showError(message) {
        $("error").textContent = message;
        $("error").classList.remove("hidden");
    }

    function clearError() {
        $("error").textContent = "";
        $("error").classList.add("hidden");
    }

    function startCategory(cat) {
        category = cat;
        page("quiz");
        $("category-label").textContent = cat;
        $("player-name").value = player;
        $("question-area").classList.add("hidden");
        $("start-area").classList.remove("hidden");
        clearError();
    }

    async function invokePython(name, args) {
        const response = await google.colab.kernel.invokeFunction(name, args, {});

        if (response && response.data) {
            if (response.data["application/json"] !== undefined) {
                return response.data["application/json"];
            }
            if (response.data["text/plain"] !== undefined) {
                let t = response.data["text/plain"];
                try {
                    return JSON.parse(t);
                } catch (e) {
                    throw new Error("Unexpected response from Python: " + String(t).slice(0, 120));
                }
            }
        }
        return response;
    }

    async function beginQuiz() {
        if (loading) return;
        clearError();

        player = $("player-name").value.trim();

        if (!player) {
            showError("Please enter your name before starting.");
            $("player-name").focus();
            return;
        }

        loading = true;
        const btn = $("begin");
        const oldText = btn.textContent;
        btn.disabled = true;
        btn.textContent = "Loading questions...";

        try {
            const response = await invokePython("quiz.questions", [category]);

            if (!response || !response.success || !response.questions || !response.questions.length) {
                throw new Error((response && response.error) || "No questions returned.");
            }

            questions = response.questions;
        } catch (e) {
            showError("Unable to load questions: " + (e.message || e));
            btn.disabled = false;
            btn.textContent = oldText;
            loading = false;
            return;
        }

        btn.disabled = false;
        btn.textContent = oldText;
        loading = false;

        index = 0;
        score = 0;
        selected = null;
        checked = false;
        finished = false;

        $("start-area").classList.add("hidden");
        $("question-area").classList.remove("hidden");

        renderQuestion();
    }

    function renderQuestion() {
        const q = questions[index];

        selected = null;
        checked = false;

        $("question-number").textContent = "Question " + (index + 1) + " of " + questions.length;
        $("score-display").textContent = "Score: " + score;
        $("progress").style.width = (index / questions.length * 100) + "%";

        $("question").textContent = q.q;
        $("options").innerHTML = "";

        $("feedback").className = "hidden";
        $("feedback").textContent = "";

        q.o.forEach((option, i) => {
            const button = document.createElement("button");
            button.type = "button";
            button.className = "option";

            const letter = document.createElement("span");
            letter.className = "option-letter";
            letter.textContent = String.fromCharCode(65 + i);

            const text = document.createElement("span");
            text.textContent = option;

            button.appendChild(letter);
            button.appendChild(text);

            button.addEventListener("click", () => {
                if (checked) return;
                selected = i;
                [...$("options").children].forEach((b, j) => {
                    b.classList.toggle("selected", i === j);
                });
            });

            $("options").appendChild(button);
        });

        $("check").disabled = false;
        $("check").classList.remove("hidden");
        $("next").classList.add("hidden");
    }

    function checkAnswer() {
        if (checked) return;

        if (selected === null) {
            $("feedback").textContent = "Please select an answer first.";
            $("feedback").className = "feedback bad";
            return;
        }

        checked = true;

        const q = questions[index];
        const correct = selected === q.a;

        if (correct) score++;

        [...$("options").children].forEach((button, i) => {
            button.disabled = true;
            if (i === q.a) button.classList.add("correct");
            else if (i === selected) button.classList.add("wrong");
        });

        $("feedback").textContent =
            (correct ? " Correct! " : "Not quite. ") +
            "Answer: " + q.o[q.a] + " — " + q.e;

        $("feedback").className = "feedback " + (correct ? "good" : "bad");

        $("score-display").textContent = "Score: " + score;
        $("progress").style.width = ((index + 1) / questions.length * 100) + "%";

        $("check").classList.add("hidden");
        $("next").classList.remove("hidden");

        $("next").textContent =
            index === questions.length - 1 ? "View My Results →" : "Next Question →";
    }

    function nextQuestion() {
        if (!checked) return;

        if (index < questions.length - 1) {
            index++;
            renderQuestion();
        } else {
            finishQuiz();
        }
    }

    function saveLocalResult(result) {
        const existing = readStorage(LOCAL_RESULTS, []);
        const records = Array.isArray(existing) ? existing : [];
        records.push(result);
        writeStorage(LOCAL_RESULTS, records);
    }

    async function finishQuiz() {
        if (finished) return;
        finished = true;

        const total = questions.length;
        const accuracy = Math.round(score / total * 100);

        $("result-score").textContent = score + "/" + total;
        $("correct").textContent = score;
        $("wrong").textContent = total - score;
        $("accuracy").textContent = accuracy + "%";
        $("result-category").textContent = "Category: " + category;

        let title = "Good effort!";
        let message = "Keep practising and you will improve.";

        if (accuracy === 100) { title = "Perfect Score!";
            message = "Fantastic! You answered every question correctly.";
        } else if (accuracy >= 80) { title = "Excellent Work!";
            message = "You have a strong understanding of this topic.";
        } else if (accuracy >= 60) { title = "Well Done!";
            message = "Nice work! Keep learning and practising.";
        }

        $("result-title").textContent = title;
        $("result-message").textContent = message;

        page("results");

        const result = {
            Name: player,
            Category: category,
            Score: score,
            "Total Questions": total,
            "Accuracy (%)": accuracy
        };

        $("save-status").textContent = "Saving your result...";

        try {
            const response = await invokePython("quiz.save", [player, category, score, total]);

            if (response && response.success) {
                $("save-status").textContent = " Your result has been saved to quiz_results.csv.";
            } else {
                saveLocalResult(result);
                $("save-status").textContent = " CSV saving failed, so your result was kept in this browser only.";
            }
        } catch (e) {
            saveLocalResult(result);
            $("save-status").textContent = " Python saving was unavailable, so your result was kept in this browser only.";
        }

        refreshDashboard();
    }

    function renderLeaderboard(records) {
        const container = $("leaderboard-table");
        container.innerHTML = "";

        if (!records.length) {
            container.innerHTML = '<p class="muted">No saved results yet. Complete a quiz first!</p>';
            return;
        }

        const wrap = document.createElement("div");
        wrap.className = "table-wrap";

        const table = document.createElement("table");
        const thead = document.createElement("thead");
        const headRow = document.createElement("tr");

        ["Rank", "Player", "Category", "Score", "Accuracy"].forEach(label => {
            const th = document.createElement("th");
            th.textContent = label;
            headRow.appendChild(th);
        });

        thead.appendChild(headRow);
        table.appendChild(thead);

        const tbody = document.createElement("tbody");

        records.slice(0, 10).forEach((r, i) => {
            const tr = document.createElement("tr");

            const values = [
                i + 1,
                r.Name ?? r.name ?? "Player",
                r.Category ?? r.category ?? "-",
                (r.Score ?? r.score ?? 0) + "/" + (r["Total Questions"] ?? r.total ?? 5),
                (r["Accuracy (%)"] ?? r.accuracy ?? 0) + "%"
            ];

            values.forEach(value => {
                const td = document.createElement("td");
                td.textContent = String(value);
                tr.appendChild(td);
            });

            tbody.appendChild(tr);
        });

        table.appendChild(tbody);
        wrap.appendChild(table);
        container.appendChild(wrap);
    }

    async function refreshDashboard() {
        let records = null;

        // The CSV (Python) is the single source of truth.
        try {
            const response = await invokePython("quiz.leaderboard", []);
            if (response && response.success && Array.isArray(response.data)) {
                records = response.data;
            }
        } catch (e) {}

        // Browser storage is used ONLY if Python is unavailable.
        if (records === null) {
            const local = readStorage(LOCAL_RESULTS, []);
            records = Array.isArray(local) ? local.slice() : [];
            records.sort((a, b) => {
                const aa = Number(a["Accuracy (%)"] ?? 0);
                const ba = Number(b["Accuracy (%)"] ?? 0);
                if (ba !== aa) return ba - aa;
                return Number(b.Score ?? 0) - Number(a.Score ?? 0);
            });
        }

        renderLeaderboard(records);

        let analytics = null;

        try {
            const response = await invokePython("quiz.analytics", []);
            if (response && response.success) analytics = response;
        } catch (e) {}

        if (!analytics) {
            const accuracies = records.map(r => Number(r["Accuracy (%)"] ?? 0));
            analytics = {
                games: records.length,
                average: accuracies.length ? accuracies.reduce((a, b) => a + b, 0) / accuracies.length : 0,
                best: accuracies.length ? Math.max(...accuracies) : 0
            };
        }

        $("games").textContent = analytics.games ?? 0;
        $("average").textContent = Number(analytics.average ?? 0).toFixed(1) + "%";
        $("best").textContent = Number(analytics.best ?? 0).toFixed(1) + "%";
    }

    // Navigation buttons
    root.querySelectorAll("[data-page]").forEach(button => {
        button.addEventListener("click", () => {
            const name = button.dataset.page;

            if (name === "categories") {
                page("home");
                $("categories").scrollIntoView({behavior: "smooth", block: "start"});
            } else {
                page(name);
            }
        });
    });

    // Category buttons
    root.querySelectorAll("[data-category]").forEach(button => {
        button.addEventListener("click", () => startCategory(button.dataset.category));
    });

    $("begin").addEventListener("click", beginQuiz);
    $("check").addEventListener("click", checkAnswer);
    $("next").addEventListener("click", nextQuestion);
    $("play-again").addEventListener("click", () => startCategory(category));
    $("refresh").addEventListener("click", refreshDashboard);

    $("player-name").addEventListener("keydown", event => {
        if (event.key === "Enter") beginQuiz();
    });

    page("home");
    refreshDashboard();

})();
</script>

</div>
"""

display(HTML(html_code))